# Machine Learning for Applied Research

## 11: Limits & Limitations of ML, and Capstone Project

In this notebook, we will cover:
- Extrapolation vs. interpolation, and why models fail differently outside their training range
- Causation vs. correlation, and what a model can and cannot tell you
- Integrating domain knowledge throughout an ML workflow
- Capstone 1: a complete pipeline, end to end, on the Bike Sharing dataset
- Capstone 2: the same workflow on messier data - loading from a raw URL, handling real missing values, and scaling for a model that needs it

### Introduction

This is the last notebook of the seminar. Before the capstone, we need to be honest about what machine learning cannot do; understanding these limits matters as much as the techniques themselves for using ML responsibly in research. The capstone that follows then brings together problem framing, model training, evaluation, cross-validation, tuning, pipelines, and interpretability into one complete workflow.

### Setup


In [ ]:
!pip install ucimlrepo --quiet

from ucimlrepo import fetch_ucirepo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

bikes = fetch_ucirepo(id=275)
feature_cols = ["season", "yr", "mnth", "hr", "holiday", "weekday",
                 "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"]
bike_df = pd.concat([bikes.data.features[feature_cols], bikes.data.targets[["cnt"]]], axis=1)

print(f"Shape: {bike_df.shape}")
print(f"Temp range: {bike_df['temp'].min():.2f} to {bike_df['temp'].max():.2f}")

### Extrapolation vs. Interpolation

A model only ever learns from the specific region of feature space its training data covered. 
- **Interpolation** = predicting within that observed range, is generally reliable.
- **Extrapolation** = predicting outside it, is risky: the model has no evidence at all about what happens out there, only whatever pattern it happened to learn inside the range it did see.

Different model families extrapolate differently, and both ways are worth understanding. Let's see this directly: train on cooler-to-moderate days only, and see how two models handle the hottest days they never encountered during training.

In [ ]:
extrap_train = bike_df[bike_df["temp"] < 0.6]
extrap_test = bike_df[bike_df["temp"] >= 0.6]

X_extrap_train = extrap_train[["temp"]]
y_extrap_train = extrap_train["cnt"]
X_extrap_test = extrap_test[["temp"]]
y_extrap_test = extrap_test["cnt"]

linreg_extrap = LinearRegression().fit(X_extrap_train, y_extrap_train)
forest_extrap = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(X_extrap_train, y_extrap_train)

temp_range = np.linspace(0, 1, 200).reshape(-1, 1)

plt.figure(figsize=(9, 5))
plt.scatter(X_extrap_train, y_extrap_train, s=5, alpha=0.3, label="Training data (temp < 0.6)")
plt.scatter(X_extrap_test, y_extrap_test, s=5, alpha=0.3, color="orange", label="Unseen hot days (temp >= 0.6)")
plt.plot(temp_range, linreg_extrap.predict(temp_range), color="black", label="Linear Regression")
plt.plot(temp_range, forest_extrap.predict(temp_range), color="red", label="Random Forest")
plt.axvline(0.6, linestyle="--", color="gray")
plt.xlabel("Normalized temperature")
plt.ylabel("Rental count")
plt.legend()
plt.title("Extrapolating beyond the training range")
plt.show()

print(f"Linear Regression - Test RMSE on unseen hot days: {rmse(y_extrap_test, linreg_extrap.predict(X_extrap_test)):.1f}")
print(f"Random Forest      - Test RMSE on unseen hot days: {rmse(y_extrap_test, forest_extrap.predict(X_extrap_test)):.1f}")

#### 1.1 Exercises:
> (a) Create a standard random 75/25 train/test split of `bike_df[["temp"]]` and `bike_df["cnt"]` (not restricted by temperature this time), fit fresh `LinearRegression` and `RandomForestRegressor` models, and compute test RMSE for both. This is the ordinary "interpolation" baseline.
>
> (b) Compare these RMSEs to the extrapolation RMSEs computed above.
>
> (c) Looking at the plot (not just the RMSE numbers), describe the qualitative difference between how the linear model and the Random Forest failed on the unseen hot days.

In [ ]:
# Your solution:





##### Solution:

In [ ]:
X_temp_train, X_temp_test, y_temp_train, y_temp_test = train_test_split(
    bike_df[["temp"]], bike_df["cnt"], test_size=0.25, random_state=42
)

linreg_normal = LinearRegression().fit(X_temp_train, y_temp_train)
forest_normal = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(X_temp_train, y_temp_train)

print(f"Linear Regression - Interpolation Test RMSE: {rmse(y_temp_test, linreg_normal.predict(X_temp_test)):.1f}")
print(f"Random Forest      - Interpolation Test RMSE: {rmse(y_temp_test, forest_normal.predict(X_temp_test)):.1f}")

# (c) The linear model keeps extending its straight line past temp=0.6,
# continuing to predict ever-higher demand as temperature rises, whether
# or not that reflects reality at the extremes. The Random Forest cannot
# predict outside the range of target values it saw in training at all.
# Past temp=0.6 its prediction flattens out at whatever value the
# nearest training leaf contained, regardless of how much hotter it
# gets. Two different failure modes, but both are guesses dressed up as
# predictions.

#### 1.2 Exercises:
> (a) Repeat the same extrapolation setup, but split on `hum` (humidity) instead of `temp`: train on `hum < 0.7`, test on `hum >= 0.7`. Refit both models on just the `hum` feature.
>
> (b) Compute test RMSE on the unseen high-humidity days for both models, and compare to an ordinary random-split baseline on `hum` (like Exercise 1.1(a), but using `hum` instead of `temp`).
>
> (c) Is the extrapolation penalty (the gap between the ordinary baseline and the extrapolation RMSE) bigger or smaller here than it was for `temp`? Would you expect every feature to extrapolate equally badly, or does it depend on how strong that feature's relationship with the target is in the first place?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
extrap_train_hum = bike_df[bike_df["hum"] < 0.7]
extrap_test_hum = bike_df[bike_df["hum"] >= 0.7]

linreg_hum = LinearRegression().fit(extrap_train_hum[["hum"]], extrap_train_hum["cnt"])
forest_hum = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42).fit(
    extrap_train_hum[["hum"]], extrap_train_hum["cnt"]
)

print(f"Linear Regression - Extrapolation Test RMSE (hum): {rmse(extrap_test_hum['cnt'], linreg_hum.predict(extrap_test_hum[['hum']])):.1f}")
print(f"Random Forest      - Extrapolation Test RMSE (hum): {rmse(extrap_test_hum['cnt'], forest_hum.predict(extrap_test_hum[['hum']])):.1f}")

X_hum_train, X_hum_test, y_hum_train, y_hum_test = train_test_split(
    bike_df[["hum"]], bike_df["cnt"], test_size=0.25, random_state=42
)
linreg_hum_normal = LinearRegression().fit(X_hum_train, y_hum_train)
print(f"Linear Regression - Interpolation Test RMSE (hum): {rmse(y_hum_test, linreg_hum_normal.predict(X_hum_test)):.1f}")

# (c) The extrapolation penalty tends to be smaller for hum than for
# temp, simply because hum has a weaker relationship with rental count
# to begin with. There is less of a strong trend for the linear model
# to (over)extend in the first place. This is a useful general
# intuition: extrapolation risk is not a fixed property of "the model",
# it depends on how strongly the specific feature drives the target:
# strong drivers extrapolate riskily, weak ones are comparatively safer
# to push slightly beyond their training range.

### Causation vs. Correlation

A model finds patterns of association in data. It does not know *why* those patterns exist, and cannot on its own distinguish "X causes Y" from "X and Y share some hidden common cause" from "this association just happened to hold in this particular dataset."

In [ ]:
print(bike_df[["hum", "weathersit", "cnt"]].corr())

Humidity is negatively correlated with rental count, but is that because humidity itself discourages people from biking, or because humid conditions tend to co-occur with `weathersit` (rain, poor visibility) more broadly, which is the actual deterrent? The correlation table shows humidity and weathersit moving together, which makes the two hard to disentangle from this data alone. This matters practically: if a city wanted to know whether covered bike-share stations (which reduce riders' exposure to rain, but do not change humidity itself) would help ridership on humid days, this model could not answer that question directly, it only knows that humidity and low ridership tend to co-occur, not which variable is doing the causal work.

#### 1.3 Exercises:
> For each scenario below, decide whether the relationship described is most likely (i) a direct causal effect, (ii) confounded by a shared underlying cause, or (iii) not distinguishable from correlation alone with the information given:
>
> (a) Ice cream sales and drowning incidents both rise in summer months.
>
> (b) In a materials dataset, higher curing temperature is associated with higher compressive strength, and a controlled lab experiment (varying only temperature, holding everything else fixed) confirms the same pattern.
>
> (c) In the AI4I dataset, higher tool wear is associated with higher failure rates.
>
> (d) In the bike-sharing dataset, higher humidity is associated with lower rental counts.

In [ ]:
# Your solution:






##### Solution:


In [ ]:
# (a) Confounded. Summer heat drives both ice cream purchases and
# swimming (and therefore drowning risk); neither directly causes the
# other.

# (b) Causal. A controlled experiment that varies only the suspected
# cause while holding everything else fixed is exactly what lets you
# move from correlation to a causal claim, it is the one thing
# observational correlational data alone cannot provide.

# (c) Plausibly causal, and mechanistically sensible (worn tools generate
# more heat and imprecision), but the AI4I dataset alone, without a
# controlled experiment that specifically varies tool wear, only
# supports "associated with," not "confirmed causal." Domain knowledge
# about the physical wear mechanism is doing a lot of the work in making
# this plausible.

# (d) Not distinguishable from correlation alone. Humidity is entangled
# with broader weather conditions in this dataset, as the correlation
# check above showed, we cannot say from this data whether humidity
# itself discourages riding, or whether it is simply a marker for worse
# conditions overall that are doing the actual discouraging.

### Integrating Domain Knowledge

Domain knowledge has done real work throughout this entire course, usually without being named as such:

- **Before modeling**: framing a research question and choosing plausible features
- **During modeling**: choosing appropriate model complexity and metrics, and excluding leakage-prone features because you understood what they actually measured
- **After modeling**: sanity-checking learned relationships against what you already know is physically or mechanistically plausible

None of this is optional polish! A model that fits the data well but contradicts basic domain understanding is a model to distrust, regardless of its test score. The capstone below is your chance to apply all of this together, deliberately, in one workflow.

#### 1.4 Exercises (discussion):
> (a) Think back over the whole three days: name one moment where domain knowledge (yours, or something built into a notebook) changed a decision that a purely statistical view of the data would not have caught on its own.
>
> (b) For your own research: what is one piece of domain knowledge you have that a model trained on your data would have no way of knowing unless you built it in deliberately (as a feature, a constraint, or a sanity check)?

In [ ]:
# Your solution:





##### Solution:

In [ ]:
# Discussion exercise, no fixed answer. Good examples from this course
# include: excluding AI4I's failure-mode columns (Notebook 5) required
# knowing what those columns actually recorded, not just that they
# correlated with the target; and the partial dependence sanity check in
# Notebook 10 only works if you already have an expectation, from domain
# knowledge, of which direction a relationship "should" go.

### Capstone 1: A Complete ML Pipeline for Bike Rental Demand

In [ ]:
X_bike = bike_df.drop(columns=["cnt"])
y_bike = bike_df["cnt"]

X_bike_train, X_bike_test, y_bike_train, y_bike_test = train_test_split(
    X_bike, y_bike, test_size=0.25, random_state=42
)
print(f"Train: {X_bike_train.shape[0]} | Test: {X_bike_test.shape[0]}")

#### 2.1 Capstone, Stage 1 - Framing:
> (a) Write a `problem_framing` dictionary for predicting hourly bike rental demand, following the template from Notebook 1.
>
> (b) Of the 12 features (`season` through `windspeed`), identify which are categorical/coded rather than genuinely continuous. Would any benefit from the `ColumnTransformer` approach from Notebook 9? (Hint: `season`, `weathersit`, `mnth`, `weekday`, and `hr` are stored as integers but do not have a numeric scale the way `temp` does.)

In [ ]:
# Your solution:





#### Solution:


In [ ]:
problem_framing_bikes = {
    "research_question": "Can we predict hourly bike rental demand from weather and calendar information?",
    "task_type": "regression",
    "target_variable": "cnt (hourly rental count)",
    "candidate_features": ["season", "yr", "mnth", "hr", "holiday", "weekday",
                            "workingday", "weathersit", "temp", "atemp", "hum", "windspeed"],
    "evaluation_metric": "RMSE, since large demand-prediction errors are more operationally costly than small ones",
    "constraints": "~17,400 rows, but with strong seasonal/hourly patterns rather than independent observations"
}
for key, value in problem_framing_bikes.items():
    print(f"{key:20s}: {value}")

# (b) season, weathersit, mnth, weekday, and hr are categorical or
# cyclical codes rather than continuous scales: "hour 23" is
# not "more" than "hour 1" in any meaningful sense, it just wraps back
# around. Treating them as plain numbers, as we do for simplicity below,
# works reasonably well for tree-based models, which only care about
# split thresholds, but could be encoded more carefully (one-hot,
# or a cyclical sine/cosine transform) for a linear model, extending
# Notebook 9's ColumnTransformer approach.

#### 2.2 Capstone, Stage 2 - Baseline Model and Metrics:
> (a) Fit a baseline `RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)` on `X_bike_train`, `y_bike_train`.
>
> (b) Compute MAE, RMSE, and R² on the test set.
>
> (c) In one sentence, interpret the R² value in plain language, for someone unfamiliar with the metric.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from sklearn.metrics import mean_absolute_error, r2_score

baseline_model = RandomForestRegressor(n_estimators=200, max_depth=8, random_state=42)
baseline_model.fit(X_bike_train, y_bike_train)

baseline_preds = baseline_model.predict(X_bike_test)
print(f"MAE:  {mean_absolute_error(y_bike_test, baseline_preds):.1f}")
print(f"RMSE: {rmse(y_bike_test, baseline_preds):.1f}")
print(f"R2:   {r2_score(y_bike_test, baseline_preds):.3f}")

# (c) An R2 of, say, 0.85 means the model's predictions account for about
# 85% of how much rental counts actually vary hour to hour, the
# remaining 15% is driven by factors the model does not have access to
# (special events, data noise, or genuinely unpredictable variation).

#### 2.3 Capstone, Stage 3 - Cross-Validated Tuning:
> (a) Build a `RandomizedSearchCV` over `RandomForestRegressor` hyperparameters of your choice (for example `n_estimators`, `max_depth`, and `min_samples_leaf`), with `cv=5`, `scoring="neg_root_mean_squared_error"`, and `random_state` set on both the model and the search.
>
> (b) Fit it on `X_bike_train`, `y_bike_train`, and report the best parameters and best CV RMSE.
>
> (c) Compare the tuned model's test RMSE to your Stage 2 baseline's test RMSE.

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint

param_distributions = {
    "n_estimators": randint(100, 400),
    "max_depth": randint(4, 16),
    "min_samples_leaf": randint(1, 10),
}

random_search_bike = RandomizedSearchCV(
    RandomForestRegressor(random_state=42),
    param_distributions=param_distributions,
    n_iter=15, cv=5, scoring="neg_root_mean_squared_error",
    random_state=42, n_jobs=-1,
)
random_search_bike.fit(X_bike_train, y_bike_train)

print(f"Best parameters: {random_search_bike.best_params_}")
print(f"Best CV RMSE: {-random_search_bike.best_score_:.1f}")

tuned_model = random_search_bike.best_estimator_
tuned_test_rmse = rmse(y_bike_test, tuned_model.predict(X_bike_test))
baseline_test_rmse = rmse(y_bike_test, baseline_model.predict(X_bike_test))

print(f"\nBaseline (Stage 2) - Test RMSE: {baseline_test_rmse:.1f}")
print(f"Tuned (Stage 3)     - Test RMSE: {tuned_test_rmse:.1f}")

#### 2.4 Capstone, Stage 4 - Interpretability and a Final Honest Report:
> (a) Compute permutation importance for `tuned_model` on the test set, scoring on `"neg_root_mean_squared_error"`, and report the top 3 most important features.
>
> (b) Do the top features make practical sense to you? Briefly justify.
>
> (c) Write a short (3-5 sentence) summary as if for a research report: state the task, the final model and its tuned settings, its test-set performance, its most important features, and one limitation from this notebook (extrapolation, or causation vs. correlation) that a reader should keep in mind.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
from sklearn.inspection import permutation_importance

perm_result_bike = permutation_importance(
    tuned_model, X_bike_test, y_bike_test, n_repeats=10, random_state=42,
    scoring="neg_root_mean_squared_error"
)

bike_importance_df = pd.DataFrame({
    "feature": X_bike_test.columns,
    "importance": perm_result_bike.importances_mean,
}).sort_values("importance", ascending=False)

print(bike_importance_df.head(3))

# (b) Hour of day, temperature, and year are typical top features here,
# all practically sensible: rentals cluster around commute hours,
# warmer weather encourages biking, and 2012 had higher overall ridership
# than 2011 as the system matured and grew its user base.

# (c) Example summary:
# "We trained a Random Forest regressor to predict hourly bike rental
# demand in Washington DC from weather and calendar features (n=17,379).
# After tuning via 5-fold cross-validated random search, the model
# achieved a test RMSE of approximately [value] rentals per hour on data
# held out from both training and tuning. The most influential features
# were hour of day, temperature, and year, consistent with commute
# patterns and weather being primary demand drivers. Two limitations are
# worth flagging: the model was trained on 2011-2012 data only and should
# not be trusted to extrapolate to substantially different conditions
# (e.g. an unprecedented heatwave), and several features, humidity in
# particular, are correlated with broader weather conditions in ways
# that make causal interpretation of any single feature's effect
# unreliable."

#### Challenge:
> In Capstone Stage 1(b), you noted that `hr` is cyclical rather than a genuine linear scale. Engineer two new features, `hr_sin = np.sin(2 * np.pi * X["hr"] / 24)` and `hr_cos = np.cos(2 * np.pi * X["hr"] / 24)`, for both the train and test sets, dropping the original `hr` column. Compare a `LinearRegression` fit with raw `hr` against one using `hr_sin`/`hr_cos` instead. Does the cyclical encoding help the linear model? Would you expect it to help the Random Forest much - why or why not?

In [ ]:
# Your solution:




##### Solution:


In [ ]:
def add_cyclical_hour(X):
    X = X.copy()
    X["hr_sin"] = np.sin(2 * np.pi * X["hr"] / 24)
    X["hr_cos"] = np.cos(2 * np.pi * X["hr"] / 24)
    return X.drop(columns=["hr"])

X_bike_train_cyclical = add_cyclical_hour(X_bike_train)
X_bike_test_cyclical = add_cyclical_hour(X_bike_test)

linreg_raw = LinearRegression().fit(X_bike_train, y_bike_train)
linreg_cyclical = LinearRegression().fit(X_bike_train_cyclical, y_bike_train)

print(f"Linear Regression, raw hr      - Test RMSE: {rmse(y_bike_test, linreg_raw.predict(X_bike_test)):.1f}")
print(f"Linear Regression, cyclical hr - Test RMSE: {rmse(y_bike_test, linreg_cyclical.predict(X_bike_test_cyclical)):.1f}")

# The cyclical encoding typically helps the linear model noticeably: a
# raw hr column forces a single straight-line effect across 0-23, which
# cannot represent demand rising toward evening and falling toward
# midnight in a way that wraps back around to a similar value near
# hour 0. sin/cos features let a linear model express exactly that kind
# of cyclic pattern directly. A Random Forest would likely change far
# less, since it can already carve the raw hr column into arbitrary
# threshold-based groups (e.g. "is hr between 7 and 9?") without needing
# the values to sit on a genuinely continuous, wrap-around scale.

### Capstone 2: Fuel Efficiency Prediction with a Messier, Real-World Dataset

Capstone 1 worked with data that arrived already clean, loaded through a Python package, and a model (Random Forest) that never needed scaling. Real datasets are rarely this well-behaved. This second capstone works through the same overall workflow (framing, cleaning, training, evaluating, reporting) but this time with messier data.

We will use the classic **Auto MPG** dataset: 398 cars from the 1970s-80s, predicting fuel efficiency (miles per gallon) from engineering specifications - cylinders, displacement, horsepower, weight, acceleration, model year, and country of origin.

#### 3.1 Capstone 2, Stage 1 - Framing:
> Write a `problem_framing` dictionary for predicting fuel efficiency, following the by-now-familiar template. Keep it brief.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
problem_framing_mpg = {
    "research_question": "Can we predict a car's fuel efficiency (mpg) from its engineering specifications?",
    "task_type": "regression",
    "target_variable": "mpg",
    "candidate_features": ["cylinders", "displacement", "horsepower", "weight", "acceleration", "model_year", "origin"],
    "evaluation_metric": "RMSE (mpg)",
    "constraints": "small dataset (398 rows), 1970s-80s vehicles only - extrapolating to modern cars would be risky (see this notebook's earlier section)"
}
for key, value in problem_framing_mpg.items():
    print(f"{key:20s}: {value}")

#### 3.2 Capstone 2, Stage 2 - Load and Clean:
> (a) Load the dataset directly from the URL below using `pd.read_csv(url, delim_whitespace=True, header=None, names=column_names, na_values="?")`, where:
> ```python
> url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
> column_names = ["mpg", "cylinders", "displacement", "horsepower", "weight",
>                 "acceleration", "model_year", "origin", "car_name"]
> ```
>
> (b) Check `.isnull().sum()` - which column has missing values, and how many?
>
> (c) Decide how to handle them: drop the affected rows, or impute (e.g. with the median). With only 398 rows total, justify your choice.
>
> (d) Drop the `car_name` column, a unique identifier, not a predictive feature.

In [ ]:
# Your solution:




##### Solution:


In [ ]:
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data"
column_names = ["mpg", "cylinders", "displacement", "horsepower", "weight",
                "acceleration", "model_year", "origin", "car_name"]

auto_df = pd.read_csv(url, delim_whitespace=True, header=None, names=column_names, na_values="?")

print(f"Shape: {auto_df.shape}")
print(f"\nMissing values per column:\n{auto_df.isnull().sum()}")

# (c) Only 6 of 398 rows (1.5%) are affected, all in horsepower. With so
# few missing values, either dropping or imputing is defensible, we
# impute with the median here, since it costs nothing to keep every row
# in a dataset this small, and horsepower's distribution isn't wildly
# skewed.
auto_df["horsepower"] = auto_df["horsepower"].fillna(auto_df["horsepower"].median())
print(f"\nMissing values after imputation:\n{auto_df.isnull().sum()}")

# (d)
auto_df = auto_df.drop(columns=["car_name"])

#### 3.3 Capstone 2, Stage 3 - Split, Scale, and Compare:
> (a) Split `auto_df` into train/test (75/25, `random_state=42`).
>
> (b) Build two models: a `Pipeline` of `StandardScaler()` followed by `Ridge(alpha=10.0)`, and a plain (unscaled) `RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)`.
>
> (c) Compare their 5-fold cross-validated RMSE on the training data.
>
> (d) Now fit `Ridge(alpha=10.0)` directly, without the `StandardScaler` step, and compare its CV RMSE to the scaled version. How much did scaling matter here, and does that match what we discussed about why Ridge specifically needs it?

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

X_auto = auto_df.drop(columns=["mpg"])
y_auto = auto_df["mpg"]

X_auto_train, X_auto_test, y_auto_train, y_auto_test = train_test_split(
    X_auto, y_auto, test_size=0.25, random_state=42
)

ridge_scaled = Pipeline(steps=[("scaler", StandardScaler()), ("regressor", Ridge(alpha=10.0))])
forest_unscaled = RandomForestRegressor(n_estimators=200, max_depth=6, random_state=42)

ridge_scaled_scores = cross_val_score(ridge_scaled, X_auto_train, y_auto_train, cv=5, scoring="neg_root_mean_squared_error")
forest_scores = cross_val_score(forest_unscaled, X_auto_train, y_auto_train, cv=5, scoring="neg_root_mean_squared_error")

print(f"Ridge (scaled)  - Mean CV RMSE: {-ridge_scaled_scores.mean():.2f}")
print(f"Random Forest   - Mean CV RMSE: {-forest_scores.mean():.2f}")

# (d)
ridge_unscaled_scores = cross_val_score(Ridge(alpha=10.0), X_auto_train, y_auto_train, cv=5, scoring="neg_root_mean_squared_error")
print(f"\nRidge (unscaled) - Mean CV RMSE: {-ridge_unscaled_scores.mean():.2f}")
print(f"Ridge (scaled)   - Mean CV RMSE: {-ridge_scaled_scores.mean():.2f}")

# Scaling typically makes a real difference here, since Auto MPG's raw
# features span very different ranges (cylinders 3-8 vs. weight
# 1613-5140). Ridge's penalty shrinks every coefficient by a 
# comparable amount regardless of that feature's scale, 
# so an unscaled small-range feature gets penalized 
# just as harshly as a large-range one, distorting the fit.

#### 3.4 Capstone 2, Stage 4 - Tune and Report:
> (a) Use `RandomizedSearchCV` to tune `alpha` for the scaled Ridge pipeline: `from scipy.stats import uniform`, then `param_distributions={"regressor__alpha": uniform(0.01, 100)}`, with `n_iter=15`, `cv=5`, `scoring="neg_root_mean_squared_error"`, `random_state=42`.
>
> (b) Report the best `alpha` and its CV RMSE, then evaluate the tuned pipeline's RMSE on the untouched test set.
>
> (c) Write a 3-4 sentence research-report-style summary (as in Capstone 1): task, model, tuned settings, honest test performance, and one limitation worth flagging (hint: what decade are these cars from?).

In [ ]:
# Your solution:





##### Solution:


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import uniform

param_distributions = {"regressor__alpha": uniform(0.01, 100)}

random_search_mpg = RandomizedSearchCV(
    ridge_scaled, param_distributions=param_distributions,
    n_iter=15, cv=5, scoring="neg_root_mean_squared_error", random_state=42, n_jobs=-1
)
random_search_mpg.fit(X_auto_train, y_auto_train)

print(f"Best parameters: {random_search_mpg.best_params_}")
print(f"Best CV RMSE: {-random_search_mpg.best_score_:.2f}")

tuned_mpg_model = random_search_mpg.best_estimator_
test_rmse_mpg = rmse(y_auto_test, tuned_mpg_model.predict(X_auto_test))
print(f"Test RMSE (held out): {test_rmse_mpg:.2f}")

# (c) Example summary:
# "We trained a Ridge regression (tuned via 5-fold cross-validated
# random search) to predict fuel efficiency from vehicle specifications,
# achieving a held-out test RMSE of approximately [value] mpg. Feature
# scaling was essential here, unlike in the Bike Sharing capstone,
# because Ridge's penalty is scale-sensitive to raw feature ranges. One
# important limitation: every vehicle in this dataset is from
# 1970-1982, so this model should not be trusted to predict fuel
# efficiency for a modern hybrid or electric vehicle: a clear case of
# extrapolating well outside the training distribution."

## Conclusion

This notebook closed the seminar by being explicit about where machine learning stops being trustworthy, and then practicing the full workflow end to end:

- **Extrapolation vs. interpolation**: models are only reliable within the range of data they were trained on, and different model families fail differently outside it
- **Causation vs. correlation**: a model can tell you what moves together, never on its own & why controlled experiments and domain knowledge are what license a causal claim
- **Domain knowledge, throughout**: framing, feature choices, leakage exclusions, and sanity-checks all depend on understanding the subject matter, not just the algorithm
- **Two capstones**: a clean, package-loaded dataset with a scale-insensitive model (Bike Sharing, Random Forest), and a messier, URL-loaded dataset with real missing values and a model that needs scaling (Auto MPG, Ridge). Different enough to show that "the workflow" isn't one-size-fits-all in its details, even though the overall shape (frame, split, train, evaluate, tune, report honestly) stays the same
